# Overflow prediction: tables for per-dataset and merged probe results

This notebook builds two ROC-AUC comparison tables for overflow prediction:

- **Table 1 — Main results**
- **Table 2 — Internal ablation**

Updated behavior:
- each column is configured manually
- supports single datasets and arbitrary merged runs
- supports different experiment names per column
- supports optional feature-only baselines per column

Set the path mappings in the config cell, then run all.


In [12]:
# =========================
# CONFIG
# =========================

COLUMNS = [
    {
        "key": "trivia",
        "label": "TriviaQA",
        "results_dir": "/app/overflow-detection/scripts/data_preprocessing/runs/split_trivia_moe/probe/results",
        "features_path": "/app/overflow-detection/scripts/data_preprocessing/runs/trivia_moe/features_context_metrics.jsonl",
        "experiment_name": "probing",
    },
    {
        "key": "squad",
        "label": "SQuAD",
        "results_dir": "/app/overflow-detection/scripts/data_preprocessing/runs/split_squad_moe/probe/results",
        "features_path": "/app/overflow-detection/scripts/data_preprocessing/runs/squad_moe/features_context_metrics.jsonl",
        "experiment_name": "probing",
    },
    {
        "key": "hotpot",
        "label": "HotpotQA",
        "results_dir": "/app/overflow-detection/scripts/data_preprocessing/runs/split_hotpotqa_moe/probe/results",
        "features_path": "/app/overflow-detection/scripts/data_preprocessing/runs/hotpotqa_moe/features_context_metrics.jsonl",
        "experiment_name": "probing",
    },
    # {
    #     "key": "squad_trivia",
    #     "label": "SQuAD + Trivia",
    #     "results_dir": "/app/overflow-detection/scripts/data_preprocessing/runs/split_combined2_7b/probe/results",
    #     "features_path": None,
    #     "experiment_name": "squad_trivia_combined",
    # },
    {
        "key": "all_three",
        "label": "SQuAD + Trivia + Hotpot",
        "results_dir": "/app/overflow-detection/scripts/data_preprocessing/runs/split_combined_moe/probe/results",
        "features_path": "/app/overflow-detection/scripts/data_preprocessing/runs/merged_moe/features_context_metrics.jsonl",
        "experiment_name": "probing",
    },
]


In [13]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate


def get_column_cfg(key):
    for col in COLUMNS:
        if col["key"] == key:
            return col
    raise KeyError(key)


COLUMN_ORDER = [c["key"] for c in COLUMNS]
COLUMN_LABELS = {c["key"]: c["label"] for c in COLUMNS}
DATA_COLS = [c["label"] for c in COLUMNS]


def load_dataset(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    df_features = pd.json_normalize(rows, sep=".")
    print(f"Loaded {len(df_features)} rows from {path}")
    return df_features


def load_probing_results(column_key, result_type):
    cfg = get_column_cfg(column_key)
    base_path = cfg["results_dir"]
    experiment_name = cfg["experiment_name"]

    file_mapping = {
        "xrag_no_query": f"probing_results_{experiment_name}_no_query_combined.json",
        "setting1": f"probing_results_setting1_{experiment_name}.json",
        "setting2": f"probing_results_setting2_{experiment_name}.json",
    }

    filepath = os.path.join(base_path, file_mapping[result_type])
    if not os.path.exists(filepath):
        print(f"[warn] Missing {result_type} file for {column_key}: {filepath}")
        return None

    with open(filepath, "r") as f:
        return json.load(f)


def style_auc_table(df, df_means, data_cols):
    style_df = pd.DataFrame('', index=df.index, columns=df.columns)
    for col in data_cols:
        if col not in df_means.columns:
            continue
        vals = df_means[col].dropna()
        if len(vals) == 0:
            continue
        max_val = vals.max()
        second_val = vals.nlargest(2).iloc[1] if len(vals) >= 2 else None
        for i in df.index:
            v = df_means.loc[i, col]
            if pd.isna(v):
                continue
            if v == max_val:
                style_df.loc[i, col] = 'font-weight: bold'
            elif second_val is not None and v == second_val:
                style_df.loc[i, col] = 'text-decoration: underline'
    return style_df


def evaluate_feature_set(X, y, cv_splits=5):
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=2000, random_state=42, C=1))
    ])
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)
    cv_results = cross_validate(
        pipe, X, y, cv=cv, scoring={'roc_auc': 'roc_auc'}, return_train_score=False
    )
    return {
        'auc_mean': cv_results['test_roc_auc'].mean(),
        'auc_std': cv_results['test_roc_auc'].std(),
    }


def compute_feature_results(df_features):
    saturation_types = ['spec_entropy', 'excess_kurtosis', 'hoyer']

    context_features = ['context_perplexity', 'context_length_chars', 'context_gzip_bits_per_char', 'context_length_tokens']
    preproj_saturation_features = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('preproj_metrics' in col) and (col[-2:] != '_n')]
    postproj_saturation_features = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('postproj_metrics' in col) and (col[-2:] != '_n')]
    prellm_saturation_features = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('postproj_metrics' in col or 'preproj_metrics' in col) and (col[-2:] != '_n')]
    mid_saturation_xrag_only = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('mid_metrics' in col) and ('nonxrag' not in col) and (col[-2:] != '_n')]
    mid_saturation_with_nonxrag = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('nonxrag' in col) and ('mid_group_metrics' in col) and (col[-2:] != '_n') and ('_first' not in col) and ('_last' not in col)] + mid_saturation_xrag_only
    mid_attn_features = [col for col in df_features.columns.values if ('attn_mid' in col)]
    last_saturation_xrag_only = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('last_metrics' in col) and ('nonxrag' not in col) and (col[-2:] != '_n')]
    last_saturation_with_nonxrag = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('last_group_metrics' in col) and ('nonxrag' in col) and (col[-2:] != '_n') and ('_first' not in col) and ('_last' not in col)] + last_saturation_xrag_only
    last_attn_features = [col for col in df_features.columns.values if ('attn_last' in col)]
    postllm_saturation_xrag_only = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('mid_metrics' in col or 'last_metrics' in col) and ('nonxrag' not in col) and (col[-2:] != '_n')]
    postllm_saturation_with_nonxrag = [col for col in df_features.columns.values if any(st in col for st in saturation_types) and ('mid_group_metrics' in col or 'last_group_metrics' in col) and ('nonxrag' in col) and (col[-2:] != '_n') and ('_first' not in col) and ('_last' not in col)] + postllm_saturation_xrag_only
    postllm_attn_features = [col for col in df_features.columns.values if ('attn' in col) and (col[-2:] != '_n')]

    features_results = {}
    for feat_name, feat_list in [
        ('context', context_features),
        ('preproj', preproj_saturation_features),
        ('postproj', postproj_saturation_features),
        ('prellm', prellm_saturation_features),
        ('mid_xrag', mid_saturation_xrag_only),
        ('mid_joint', mid_saturation_with_nonxrag),
        ('mid_attn', mid_attn_features),
        ('last_xrag', last_saturation_xrag_only),
        ('last_joint', last_saturation_with_nonxrag),
        ('last_attn', last_attn_features),
        ('postllm_xrag', postllm_saturation_xrag_only),
        ('postllm_joint', postllm_saturation_with_nonxrag),
        ('postllm_attn', postllm_attn_features),
    ]:
        if not feat_list:
            continue
        required = feat_list + ['overflow_label']
        if not all(c in df_features.columns for c in required):
            continue
        data = df_features[required].dropna()
        if len(data) == 0:
            continue
        features_results[feat_name] = evaluate_feature_set(data[feat_list].values, data['overflow_label'].values)
    return features_results


In [14]:
# ============================================================================
# TABLES: MAIN RESULTS + INTERNAL ABLATION
# ============================================================================

def generate_tables():
    columns = COLUMN_ORDER

    all_data = {}
    for col_key in columns:
        cfg = get_column_cfg(col_key)
        all_data[col_key] = {
            'probing_no_query': load_probing_results(col_key, 'xrag_no_query'),
            'probing_setting1': load_probing_results(col_key, 'setting1'),
            'probing_setting2': load_probing_results(col_key, 'setting2'),
            'features': None,
        }

        feature_path = cfg.get("features_path")
        if feature_path:
            if os.path.exists(feature_path):
                df_features = load_dataset(feature_path)
                all_data[col_key]['features'] = compute_feature_results(df_features)
            else:
                print(f"[warn] Missing features file for {col_key}: {feature_path}")

    def fmt(mean, std):
        if mean is None or std is None:
            return "---"
        return f"{mean:.3f} ± {std:.3f}"

    def get_probe_metrics(col_key, key):
        for result_type in ['probing_no_query', 'probing_setting1', 'probing_setting2']:
            data = all_data[col_key][result_type]
            if data and key in data:
                result = data[key]
                return {
                    'auc_mean': result.get('auc'),
                    'auc_std': result.get('auc_std'),
                }
        return None

    def get_feature_metrics(col_key, key):
        if all_data[col_key]['features'] and key in all_data[col_key]['features']:
            return all_data[col_key]['features'][key]
        return None

    def build_table_df(sections_config):
        rows_display = []
        rows_means = []

        for stage_name, section_rows in sections_config:
            for row_idx, (feature_name, keys) in enumerate(section_rows):
                probe_key, feature_key = keys
                row_d = {'Stage': stage_name if row_idx == 0 else '', 'Features': feature_name}
                row_m = {'Stage': stage_name if row_idx == 0 else '', 'Features': feature_name}

                for col_key in columns:
                    col = COLUMN_LABELS[col_key]
                    m = get_probe_metrics(col_key, probe_key) if probe_key else (
                        get_feature_metrics(col_key, feature_key) if feature_key else None
                    )
                    if m and m.get('auc_mean') is not None:
                        row_d[col] = fmt(m['auc_mean'], m['auc_std'])
                        row_m[col] = m['auc_mean']
                    else:
                        row_d[col] = '---'
                        row_m[col] = np.nan

                rows_display.append(row_d)
                rows_means.append(row_m)

        return pd.DataFrame(rows_display), pd.DataFrame(rows_means)

    main_results_sections = [
        ("Pre-compression", [
            ("Context", [None, 'context']),
        ]),
        ("Pre-inference", [
            ("Embedding", ['xrag_preproj+postproj_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting1_preproj+postproj_with_preproj_q+postproj_q_linear_sklearn', None]),
            ("Saturation", [None, 'prellm']),
        ]),
        ("Post-inference", [
            ("Attention", [None, 'postllm_attn']),
            ("Embedding", ['xrag_mid+last_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting2_mid+last_with_mid_q+last_q_linear_sklearn', None]),
            ("Saturation", [None, 'postllm_xrag']),
            ("Saturation-joint", [None, 'postllm_joint']),
        ]),
    ]

    internal_ablation_sections = [
        ("Pre-projection", [
            ("Embedding", ['xrag_preproj_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting1_preproj_with_preproj_q_linear_sklearn', None]),
            ("Saturation", [None, 'preproj']),
        ]),
        ("Post-projection", [
            ("Embedding", ['xrag_postproj_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting1_postproj_with_postproj_q_linear_sklearn', None]),
            ("Saturation", [None, 'postproj']),
        ]),
        ("Middle layer", [
            ("Attention", [None, 'mid_attn']),
            ("Embedding", ['xrag_mid_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting2_mid_with_mid_q_linear_sklearn', None]),
            ("Saturation", [None, 'mid_xrag']),
            ("Saturation-joint", [None, 'mid_joint']),
        ]),
        ("Last layer", [
            ("Attention", [None, 'last_attn']),
            ("Embedding", ['xrag_last_no_query_linear_sklearn', None]),
            ("Embedding-joint", ['setting2_last_with_last_q_linear_sklearn', None]),
            ("Saturation", [None, 'last_xrag']),
            ("Saturation-joint", [None, 'last_joint']),
        ]),
    ]

    table1_df, table1_means = build_table_df(main_results_sections)
    table2_df, table2_means = build_table_df(internal_ablation_sections)
    return (table1_df, table1_means), (table2_df, table2_means)


In [15]:
(table1_df, table1_means), (table2_df, table2_means) = generate_tables()

display(table1_df.style.apply(lambda _: style_auc_table(table1_df, table1_means, DATA_COLS), axis=None))
display(table2_df.style.apply(lambda _: style_auc_table(table2_df, table2_means, DATA_COLS), axis=None))


Loaded 7544 rows from /app/overflow-detection/scripts/data_preprocessing/runs/trivia_moe/features_context_metrics.jsonl
Loaded 4118 rows from /app/overflow-detection/scripts/data_preprocessing/runs/squad_moe/features_context_metrics.jsonl
Loaded 5147 rows from /app/overflow-detection/scripts/data_preprocessing/runs/hotpotqa_moe/features_context_metrics.jsonl
Loaded 16809 rows from /app/overflow-detection/scripts/data_preprocessing/runs/merged_moe/features_context_metrics.jsonl


,Stage,Features,TriviaQA,SQuAD,HotpotQA,SQuAD + Trivia + Hotpot
0,Pre-compression,Context,0.545 ± 0.020,0.570 ± 0.028,0.546 ± 0.016,0.504 ± 0.013
1,Pre-inference,Embedding,0.632 ± 0.029,0.647 ± 0.023,0.639 ± 0.007,0.764 ± 0.009
2,,Embedding-joint,0.651 ± 0.021,0.680 ± 0.029,0.697 ± 0.007,0.792 ± 0.008
3,,Saturation,0.561 ± 0.016,0.537 ± 0.020,0.533 ± 0.025,0.660 ± 0.009
4,Post-inference,Attention,0.616 ± 0.026,0.602 ± 0.010,0.619 ± 0.007,0.691 ± 0.014
5,,Embedding,0.631 ± 0.028,0.646 ± 0.021,0.634 ± 0.008,0.760 ± 0.009
6,,Embedding-joint,0.657 ± 0.022,0.682 ± 0.029,0.714 ± 0.005,0.803 ± 0.009
7,,Saturation,0.591 ± 0.011,0.577 ± 0.016,0.548 ± 0.023,0.666 ± 0.009
8,,Saturation-joint,0.612 ± 0.020,0.583 ± 0.013,0.611 ± 0.019,0.688 ± 0.008


,Stage,Features,TriviaQA,SQuAD,HotpotQA,SQuAD + Trivia + Hotpot
0,Pre-projection,Embedding,0.622 ± 0.031,0.625 ± 0.023,0.630 ± 0.008,0.753 ± 0.010
1,,Embedding-joint,0.639 ± 0.026,0.661 ± 0.028,0.687 ± 0.007,0.781 ± 0.009
2,,Saturation,0.515 ± 0.013,0.499 ± 0.017,0.507 ± 0.009,0.510 ± 0.011
3,Post-projection,Embedding,0.624 ± 0.028,0.643 ± 0.019,0.629 ± 0.007,0.752 ± 0.010
4,,Embedding-joint,0.647 ± 0.019,0.675 ± 0.028,0.688 ± 0.006,0.783 ± 0.008
5,,Saturation,0.563 ± 0.014,0.543 ± 0.024,0.535 ± 0.026,0.658 ± 0.008
6,Middle layer,Attention,0.599 ± 0.030,0.581 ± 0.018,0.605 ± 0.010,0.663 ± 0.012
7,,Embedding,0.624 ± 0.028,0.643 ± 0.019,0.629 ± 0.007,0.752 ± 0.010
8,,Embedding-joint,0.651 ± 0.020,0.684 ± 0.027,0.705 ± 0.010,0.799 ± 0.010
9,,Saturation,0.562 ± 0.014,0.544 ± 0.024,0.535 ± 0.026,0.658 ± 0.008


#### Latex Helpers

In [16]:
import os
import numpy as np

def latex_escape(s):
    s = str(s)
    return (
        s.replace("\\", r"\textbackslash{}")
         .replace("&", r"\&")
         .replace("%", r"\%")
         .replace("$", r"\$")
         .replace("#", r"\#")
         .replace("_", r"\_")
    )

def fmt_latex_cell(mean, display_value, best=False, second=False):
    if pd.isna(mean) or display_value in ["---", None]:
        return "---"

    value = str(display_value).replace(" ± ", r" $\pm$ ")

    if best:
        return r"\textbf{" + value + "}"
    if second:
        return r"\underline{" + value + "}"
    return value

def export_latex_table(
    table_df,
    table_means,
    output_path,
    caption,
    label,
    data_cols=None,
    table_env="table*",
    position="h!",
    feature_rename=None,
):
    """
    Export table_df/table_means to LaTeX booktabs format.
    Bold = best per dataset column.
    Underline = second-best per dataset column.
    """

    if data_cols is None:
        data_cols = [c for c in table_df.columns if c not in ["Stage", "Features"]]

    feature_rename = feature_rename or {}

    # best / second-best per metric column
    best_idx = {}
    second_idx = {}

    for col in data_cols:
        vals = table_means[col].astype(float)
        valid = vals.dropna().sort_values(ascending=False)

        if len(valid) > 0:
            best_idx[col] = valid.index[0]
        if len(valid) > 1:
            second_idx[col] = valid.index[1]

    n_cols = 2 + len(data_cols)
    tabular_spec = "ll" + "c" * len(data_cols)

    lines = []
    lines.append(rf"\begin{{{table_env}}}[{position}]")
    lines.append(r"\centering")
    lines.append(r"\small")
    lines.append(rf"\begin{{tabular}}{{{tabular_spec}}}")
    lines.append(r"\toprule")

    header = (
        r"\textbf{Stage} & \textbf{Features} & "
        + " & ".join([rf"\textbf{{{latex_escape(c)}}}" for c in data_cols])
        + r" \\"
    )
    lines.append(header)
    lines.append(r"\midrule")

    for i, row in table_df.iterrows():
        stage = row["Stage"]
        feature = feature_rename.get(row["Features"], row["Features"])

        # add midrule before every new non-first stage
        if i > 0 and stage != "":
            lines.append(r"\midrule")

        stage_cell = (
            rf"\multicolumn{{1}}{{l}}{{\textbf{{{latex_escape(stage)}}}}}"
            if stage != ""
            else ""
        )

        feature_cell = latex_escape(feature)

        row_cells = [stage_cell, feature_cell]

        for col in data_cols:
            mean = table_means.loc[i, col]
            display_value = row[col]

            row_cells.append(
                fmt_latex_cell(
                    mean,
                    display_value,
                    best=(best_idx.get(col) == i),
                    second=(second_idx.get(col) == i),
                )
            )

        lines.append(" & ".join(row_cells) + r" \\")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(rf"\caption{{{caption}}}")
    lines.append(rf"\label{{{label}}}")
    lines.append(rf"\end{{{table_env}}}")

    latex = "\n".join(lines)

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(latex)

    print(f"[saved] {output_path}")
    return latex

In [18]:
caption = (
    r"Ablation study examining feature extraction at different architectural stages (ROC-AUC). "
    r"\textbf{Pre-projection}: retriever embeddings before projection. "
    r"\textbf{Post-projection}: representations after projection. "
    r"\textbf{Middle layer}: intermediate LLM hidden states. "
    r"\textbf{Last layer}: final LLM hidden states. "
    r"Representation-joint combines query and context representations. "
    r"\textbf{Bold}: best performance per dataset; \underline{underlined}: second-best."
)

latex = export_latex_table(
    table_df=table2_df,
    table_means=table2_means,
    output_path="moe_internal_ablation_table.tex",
    caption=caption,
    label="tab:moe_internal_ablation",
    data_cols=DATA_COLS,
    feature_rename={
        "Embedding": "Representation",
        "Embedding-joint": "Representation-joint",
    },
)

print(latex)

[saved] moe_internal_ablation_table.tex
\begin{table*}[h!]
\centering
\small
\begin{tabular}{llcccc}
\toprule
\textbf{Stage} & \textbf{Features} & \textbf{TriviaQA} & \textbf{SQuAD} & \textbf{HotpotQA} & \textbf{SQuAD + Trivia + Hotpot} \\
\midrule
\multicolumn{1}{l}{\textbf{Pre-projection}} & Representation & 0.622 $\pm$ 0.031 & 0.625 $\pm$ 0.023 & 0.630 $\pm$ 0.008 & 0.753 $\pm$ 0.010 \\
 & Representation-joint & 0.639 $\pm$ 0.026 & 0.661 $\pm$ 0.028 & 0.687 $\pm$ 0.007 & 0.781 $\pm$ 0.009 \\
 & Saturation & 0.515 $\pm$ 0.013 & 0.499 $\pm$ 0.017 & 0.507 $\pm$ 0.009 & 0.510 $\pm$ 0.011 \\
\midrule
\multicolumn{1}{l}{\textbf{Post-projection}} & Representation & 0.624 $\pm$ 0.028 & 0.643 $\pm$ 0.019 & 0.629 $\pm$ 0.007 & 0.752 $\pm$ 0.010 \\
 & Representation-joint & \underline{0.647 $\pm$ 0.019} & \underline{0.675 $\pm$ 0.028} & \underline{0.688 $\pm$ 0.006} & \underline{0.783 $\pm$ 0.008} \\
 & Saturation & 0.563 $\pm$ 0.014 & 0.543 $\pm$ 0.024 & 0.535 $\pm$ 0.026 & 0.658 $\pm$ 0.008 